# **Scattering correction for infrared spectra of biological cells using computational infrared microspectroscopy and deep learning**

*Sergio G. Rodrigo $^{1,2*}$, Ilia L. Rasskazov $^{3}$, Luis Martin-Moreno $^{1,4}$, Martin Schnell $^{5,6,7*}$*



+ 1 Instituto de Nanociencia y Materiales de Aragón (INMA), CSIC-Universidad de Zaragoza, Zaragoza, 50009, Spain
+ 2 Departamento de Física Aplicada, Universidad de Zaragoza, Zaragoza, 50009, Spain
+ 3 Independent Researcher, San Jose, California 95124, United States
+ 4 Departamento de Física de la Materia Condensada, Universidad de Zaragoza, Zaragoza, 50009, Spain
+ 5 CIC nanoGUNE BRTA, 20018 Donostia-San Sebastián, Spain
+ 6 Donostia International Physics Center, Donostia-San Sebastian 20018, Spain
+ 7 IKERBASQUE, Basque Foundation for Science, 48013 Bilbao, Spain

$^*$ Corresponding authors. Email: sergut@unizar.es (SGR), schnelloptics@gmail.com (MS)


# *Description*

As explained in the main text (see Fig. 4(a)), two deep-learning models were trained as surrogates for the FDTD calculations of the absorbance spectrum, $A(\nu)$ , and the true absorption spectrum, $\mathcal{A}_{true} (\nu)$. Although the models were trained independently on the absorbance and true absorption spectra, respectively, they share an identical architecture and hiperparameters, described below.

The neural network consist in a fully connected feed-forward architecture and was implemented using the Keras library. Each model takes the parameter $p=(r_x,r_y,r_z,k_m )$ as input, corresponding to the three ellipsoid semi-axes and the $k_m$ parameter for scaling the imaginary refractive index of Matrigel, $k(\nu)$. The output layer is $N_{wav}=251$ vector, corresponding to one neuron per spectral wavelength value of the spectrum to be predicted.

# Main Packages

In [ ]:
# Common packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load matlab files
import scipy.io

# Neural Networks packages
import tensorflow as tf
from tensorflow.keras import models,layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import SGD, Adam

# Auxiliary Plotting Functions

In [ ]:
def plot_history_by_key(history):  
  import matplotlib.pyplot as plt
  keys=history.history.keys()
  print(keys)
  for key in keys:
    plt.loglog(history.history[key],marker='o',markersize=0.0, linewidth=1.0,label=key)
  plt.xlabel('epoch')
  plt.legend()
  plt.show()
  pass

def plot_nn_vs_true(nn_results,true_results,output_name):
  index=np.linspace(1,len(nn_results),len(nn_results))
  fig, ax = plt.subplots(2,1,dpi=100, figsize=(8,8)) #sharex='col', sharey='row',
  ax[0].plot(index,nn_results, color="blue", label="Neural Network",fillstyle='none',linewidth=0, marker='.', markersize=15)
  ax[0].plot(index,true_results, color="red", label="True values",linewidth=0, marker='.', markersize=10)
  #ax[0].set_aspect(1.0/ax[0].get_data_ratio(), adjustable='box')
  txt1 ="index"
  txt2 = output_name
  ax[0].set_xlabel(txt1)
  ax[0].set_ylabel(txt2)
  ax[0].legend()

  ax[1].plot(true_results,true_results, color="blue", fillstyle='none',linewidth=2, marker='.', markersize=0)
  ax[1].plot(nn_results,true_results, color="red", linewidth=0, marker='.', markersize=10)
  ax[1].set_aspect(1.0/ax[1].get_data_ratio(), adjustable='box')
  txt1 = output_name+' - Neural Network'
  txt2 = output_name+' - True values'
  ax[1].set_xlabel(txt1)
  ax[1].set_ylabel(txt2)
  txt="Neural Network vs True values" # Final wavelength)
  fig.suptitle(txt)
  fig.show
pass

# Load the Data

In [ ]:
# Load the Training and Validation Datasets
path="./data/training_data/"
collect = scipy.io.loadmat(path+'fdtd_simulations.mat')

# Wavelengths
wlength=collect['vec_wn']

# Spectra
absorbance=collect['collect_spec_norm'].T
absorption=collect['collect_frac_abs'].T

## INPUT: $\vec p = (r_x,r_y,r_z,k_m) $

In [ ]:
# Parameters (radii,material (k))
collect_params=collect['collect_params']

params=collect_params[0,0]
params_list=[p for p in params]
df_radii=pd.DataFrame(params_list[0])
print("radii shape=",df_radii.shape)

df_material=pd.DataFrame(params_list[1])
print('Material k shape=',df_material.shape)

materialno=params_list[2]
print('Material number shape=',materialno.shape)

# INPUT NEURAL NETWORK
df_x=pd.concat([df_radii, df_material], axis=1, ignore_index=True)
df_x.columns=['rx','ry','rz','k']

## OUTPUT: Absorbance = $A^p(\nu)$ *or* True Absorption = $\mathcal{A}^p(\nu)$  

In [ ]:
# Select the spectra for training
SPEC_OUTPUT_NAME='absorption' # "absorbance' or 'absorption'

if(SPEC_OUTPUT_NAME=='absorbance'):
    df_spec=absorbance
    
if(SPEC_OUTPUT_NAME=='absorption'):
    df_spec=absorption

df_spec=pd.DataFrame(df_spec)
df_spec_raw=collect['collect_spec'].T
df_spec_raw=pd.DataFrame(df_spec_raw)

# OUTPUT NEURAL NETWORK
df_y = df_spec

# Randomization, Normalization and Split Into Training and Validation Sets

In [ ]:
# set random seed for reproducibility
np.random.seed()

# randomize both dataframes with the same random seed
df_x = df_x.sample(frac=1,random_state=1)
df_y = df_y.sample(frac=1,random_state=1)

# Splitting training and validation sets
Ndata=df_x.shape[0]
n_train = int(Ndata*0.8)
n_val = int(Ndata-n_train)

print("Number of spectra for training:",n_train)
print("Number of spectra for validation:",n_val)

# Normalize input data
df_max = df_x.max()
max_axis=df_max[0:2].max()
df_max['rx']=max_axis;df_max['ry']=max_axis;df_max['rz']=max_axis
df_x_norm=df_x/df_max
print("\ndf_max=",df_max)

# Normalize output data
df_y_max = np.max(df_y)
df_y = df_y /df_y_max

# Set the training and validation data
x_train=np.array(df_x_norm[0:n_train])
y_train=np.array(df_y[0:n_train])
x_val=np.array(df_x_norm[n_train:n_train+n_val])
y_val=np.array(df_y[n_train:n_train+n_val])

# Train the Neural Network

In [ ]:
model_name='./surrogate_neural_network_model_'+SPEC_OUTPUT_NAME+'.keras'
print("Model name:",model_name)

In [ ]:
input_neurons=x_train.shape[1]  # One neuron for each input parameter
output_neurons=y_train.shape[1] # One neuron for wavelength to predict

batch_size = 64
epochs = 1000
hidden_layer_neurons = 64

model = Sequential()
model.add(tf.keras.layers.InputLayer((input_neurons,)))
for i in range(4):
  model.add(Dense(hidden_layer_neurons, activation='sigmoid'))
  model.add(layers.BatchNormalization())
model.add(Dense(output_neurons, activation=None))
model.summary()

# Optimizer
eta = 1.0
momentum=0.001              # Keras momentum
optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.01, momentum=momentum)

# Model compile
model.compile(loss='mse',
          optimizer=optimizer, metrics=['mse'])

# Callback
callbacks = [
tf.keras.callbacks.ModelCheckpoint(model_name,
save_best_only=True)
]

# Model fit
history = model.fit(x_train, y_train,
                    batch_size=batch_size,
                    epochs=epochs,
                    verbose=1,
                    validation_data=(x_val, y_val),callbacks=callbacks)

plot_history_by_key(history)

# Neural Network Predictions vs. Ground Truth

## True vs Ground

In [ ]:
# Load model
model_name='./surrogate_neural_network_model_'+SPEC_OUTPUT_NAME+'.keras'
model=models.load_model(model_name)

nn_results=model.predict(x_val,verbose=0)
true_results=y_val

plt.scatter(true_results,nn_results,alpha=0.1,color='r')
plt.plot(y_val,y_val,color='blue')
plt.xlabel("True values")
plt.ylabel("Predictions")
plt.show()

## Compare spectra

In [ ]:
# Load model
model_name='./surrogate_neural_network_model_'+SPEC_OUTPUT_NAME+'.keras'
model=models.load_model(model_name)

# Pick a random spectrum from the validation dataset  
j=np.random.randint(x_val.shape[0]) 
print('index:',j)
results=model.predict(x_val,verbose=0)
nn_results=results[j,:]
true_results=y_val[j,:]

# Plot predictions vs true values
plot_nn_vs_true(nn_results,true_results,SPEC_OUTPUT_NAME)